# Read a merged historical publication

Use an existing combined merge, such as the output from the Bybit Quick Start. Start from the MarketForge research kernel inside the checkout. This notebook reads existing files.

In [ ]:
from forge.datasets import MarketData

market = MarketData()
catalog = market.discover(metadata_only=True)

matches = catalog.find(operation="merge", exchange="bybit", symbol="BTCUSDT")
display(matches.summary())

Choose the publication from your run. Its directory contains the manifest shown under **Data → Merged** in Forge. The catalog lists primary files, so several rows can belong to the same publication.

In [ ]:
publication = market.open(input("Merged publication directory: ").strip())
display(publication.streams())
display(publication.schema())
display(publication.lineage())

Trades and depth can occupy the same physical file. Select events by kind, or paste a stream ID from the stream table to isolate one logical stream.

In [ ]:
trades = publication.query(data_type="trades")
depth = publication.query(data_type="depth")
display(trades.head(5, input_rows=20_000))
display(depth.head(5, input_rows=20_000))
stream = publication.select(stream_id=input("Exact stream ID: ").strip())
display(stream.membership())

Inspect physical locators before scanning. Pick one partition and read a time window within that partition. The lazy projection omits Decimal256 payload columns. Use Arrow batches when you need exact typed values.

In [ ]:
display([part.info() for part in publication.partitions()])
reference = input("Partition reference: ").strip()
start = input("Start UTC (ISO timestamp): ").strip()
end = input("End UTC (ISO timestamp): ").strip()
selection = publication.query(data_type="trades", partition=reference)
columns = ["event_timestamp_ns", "instrument_id", "stream_id"]
display(selection.explain(columns=columns, start=start, end=end))
display(selection.scan(columns=columns, start=start, end=end).limit(10).collect())